سلول ۱ — مسیرها و تنظیمات

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import gc
from tqdm.auto import tqdm

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
DATA_FEAT_DIR = PROJECT_ROOT / "Data_feat"
DATA_ML_DIR = PROJECT_ROOT / "Data_ml"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

PAIR_FEATURE_DIR = DATA_ML_DIR / "pair_features"

for d in [DATA_ML_DIR, PAIR_FEATURE_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODELS_TO_RUN = [
    "esm2_t6_8M_UR50D",
    "esm2_t12_35M_UR50D",
    "esm2_t30_150M_UR50D",
    "esm2_t33_650M_UR50D",
    "esm2_t36_3B_UR50D",
    "prot_bert",
    "prot_bert_bfd",
]

PAIRS_MODEL_READY_PATH = PAIR_DIR / "pairs_all_model_ready.csv"

pairs = pd.read_csv(PAIRS_MODEL_READY_PATH, dtype=str, low_memory=False)
pairs["label"] = pairs["label"].astype(int)

print("Original model-ready pairs:", pairs.shape)
print(pairs["label"].value_counts())
display(pairs.head())

سلول ۲ — حذف پروتئین فوق‌بلند Q8WZ42

In [ ]:
EXCLUDE_ACCESSIONS = {"Q8WZ42"}

pairs_embedding_ready = pairs[
    ~(
        pairs["enz_ac"].astype(str).isin(EXCLUDE_ACCESSIONS) |
        pairs["sub_ac"].astype(str).isin(EXCLUDE_ACCESSIONS)
    )
].copy()

print("Before:", pairs.shape)
print("After embedding-ready:", pairs_embedding_ready.shape)
print("Removed pairs:", len(pairs) - len(pairs_embedding_ready))

print("\nLabel counts:")
print(pairs_embedding_ready["label"].value_counts())

print("\nEnzyme class counts:")
print(pairs_embedding_ready["enzyme_class"].value_counts())

removed_embedding = pairs[
    (
        pairs["enz_ac"].astype(str).isin(EXCLUDE_ACCESSIONS) |
        pairs["sub_ac"].astype(str).isin(EXCLUDE_ACCESSIONS)
    )
].copy()

display(removed_embedding)

pairs_embedding_ready.to_csv(
    PAIR_DIR / "pairs_all_embedding_ready.csv",
    index=False
)

removed_embedding.to_csv(
    QC_DIR / "excluded_long_sequence_pairs_Q8WZ42.csv",
    index=False
)

سلول ۳ — توابع بارگذاری embedding

In [ ]:
def get_embedding_paths(model_name, acc):
    seq_path = DATA_FEAT_DIR / "per_sequence" / model_name / "proteins" / f"{acc}.npy"
    res_path = DATA_FEAT_DIR / "per_residue" / model_name / "proteins" / f"{acc}.npy"
    return seq_path, res_path


def load_seq_embedding(model_name, acc):
    seq_path, _ = get_embedding_paths(model_name, acc)
    if not seq_path.exists():
        raise FileNotFoundError(seq_path)
    return np.load(seq_path).astype(np.float32)


def load_res_embedding(model_name, acc):
    _, res_path = get_embedding_paths(model_name, acc)
    if not res_path.exists():
        raise FileNotFoundError(res_path)
    return np.load(res_path).astype(np.float32)


def check_model_embedding_coverage(model_name, pairs_df):
    required = sorted(
        set(pairs_df["enz_ac"].astype(str)) |
        set(pairs_df["sub_ac"].astype(str))
    )
    
    rows = []
    for acc in required:
        seq_path, res_path = get_embedding_paths(model_name, acc)
        rows.append({
            "model": model_name,
            "accession": acc,
            "has_seq": seq_path.exists(),
            "has_res": res_path.exists(),
            "ok": seq_path.exists() and res_path.exists(),
        })
    
    cov = pd.DataFrame(rows)
    return cov

سلول ۴ — QC coverage برای pairs_embedding_ready

In [ ]:
coverage_rows = []

for model_name in MODELS_TO_RUN:
    cov = check_model_embedding_coverage(model_name, pairs_embedding_ready)
    
    coverage_rows.append({
        "model": model_name,
        "n_required_proteins": len(cov),
        "n_ok": int(cov["ok"].sum()),
        "n_missing": int((~cov["ok"]).sum()),
        "coverage": float(cov["ok"].mean()),
    })

embedding_ready_coverage = pd.DataFrame(coverage_rows)
display(embedding_ready_coverage)

embedding_ready_coverage.to_csv(
    QC_DIR / "embedding_ready_coverage_all_models.csv",
    index=False
)

سلول ۵ — per-sequence pair features

In [ ]:
def make_per_sequence_pair_features(e, s):
    e = e.astype(np.float32)
    s = s.astype(np.float32)
    
    absdiff = np.abs(e - s)
    prod = e * s
    
    x = np.concatenate([e, s, absdiff, prod]).astype(np.float32)
    return x


def build_per_sequence_features_for_model(model_name, pairs_df):
    rows = []
    meta = pairs_df[[
        "pair_id", "group_id", "enzyme_class",
        "enz_ac", "sub_ac", "enz_gene", "sub_gene",
        "enzyme_type", "label", "source", "pmid"
    ]].copy()
    
    for _, r in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc=f"per-seq {model_name}"):
        e = load_seq_embedding(model_name, r["enz_ac"])
        s = load_seq_embedding(model_name, r["sub_ac"])
        x = make_per_sequence_pair_features(e, s)
        rows.append(x)
    
    X = np.vstack(rows).astype(np.float32)
    
    feature_cols = [f"{model_name}_seq_f{i}" for i in range(X.shape[1])]
    X_df = pd.DataFrame(X, columns=feature_cols)
    
    return X_df, meta

سلول ۶ — per-residue summary features کامل

In [ ]:
def cosine_similarity_matrix(A, B, eps=1e-8):
    A = A.astype(np.float32)
    B = B.astype(np.float32)
    
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + eps)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + eps)
    
    return A_norm @ B_norm.T


def safe_stats(x, prefix):
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return {
            f"{prefix}_mean": 0.0,
            f"{prefix}_std": 0.0,
            f"{prefix}_min": 0.0,
            f"{prefix}_max": 0.0,
            f"{prefix}_q10": 0.0,
            f"{prefix}_q25": 0.0,
            f"{prefix}_q50": 0.0,
            f"{prefix}_q75": 0.0,
            f"{prefix}_q90": 0.0,
        }
    
    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x)),
        f"{prefix}_min": float(np.min(x)),
        f"{prefix}_max": float(np.max(x)),
        f"{prefix}_q10": float(np.quantile(x, 0.10)),
        f"{prefix}_q25": float(np.quantile(x, 0.25)),
        f"{prefix}_q50": float(np.quantile(x, 0.50)),
        f"{prefix}_q75": float(np.quantile(x, 0.75)),
        f"{prefix}_q90": float(np.quantile(x, 0.90)),
    }


def topk_values(flat, k):
    flat = np.asarray(flat, dtype=np.float32)
    if flat.size == 0:
        return np.array([], dtype=np.float32)
    k = min(k, flat.size)
    idx = np.argpartition(flat, -k)[-k:]
    return flat[idx]


def make_residue_summary_features(E, S, tau_values=(0.3, 0.5, 0.7), topks=(5, 10, 20, 50)):
    sim = cosine_similarity_matrix(E, S)
    flat = sim.ravel()
    
    feats = {}
    
    feats.update(safe_stats(flat, "sim_all"))
    
    # top-k over all residue-residue similarities
    for k in topks:
        vals = topk_values(flat, k)
        feats.update(safe_stats(vals, f"top{k}"))
    
    # row max: for each enzyme residue, best matching substrate residue
    rowmax = sim.max(axis=1) if sim.shape[1] > 0 else np.array([], dtype=np.float32)
    colmax = sim.max(axis=0) if sim.shape[0] > 0 else np.array([], dtype=np.float32)
    
    feats.update(safe_stats(rowmax, "rowmax"))
    feats.update(safe_stats(colmax, "colmax"))
    
    for tau in tau_values:
        feats[f"rowmax_frac_gt_{tau}"] = float(np.mean(rowmax > tau)) if rowmax.size else 0.0
        feats[f"colmax_frac_gt_{tau}"] = float(np.mean(colmax > tau)) if colmax.size else 0.0
        feats[f"all_frac_gt_{tau}"] = float(np.mean(flat > tau)) if flat.size else 0.0
    
    # histogram bins
    bins = np.linspace(-1, 1, 21)
    hist, _ = np.histogram(flat, bins=bins)
    hist = hist.astype(np.float32)
    hist = hist / (hist.sum() + 1e-8)
    
    for i, v in enumerate(hist):
        feats[f"hist_bin_{i:02d}"] = float(v)
    
    # shape info
    feats["enzyme_len"] = int(E.shape[0])
    feats["substrate_len"] = int(S.shape[0])
    feats["len_ratio_e_over_s"] = float(E.shape[0] / max(S.shape[0], 1))
    feats["len_abs_diff"] = int(abs(E.shape[0] - S.shape[0]))
    
    return feats

سلول ۷ — ساخت per-residue features برای یک مدل

In [ ]:
def build_per_residue_features_for_model(model_name, pairs_df):
    feat_rows = []
    
    meta = pairs_df[[
        "pair_id", "group_id", "enzyme_class",
        "enz_ac", "sub_ac", "enz_gene", "sub_gene",
        "enzyme_type", "label", "source", "pmid"
    ]].copy()
    
    for _, r in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc=f"per-res {model_name}"):
        E = load_res_embedding(model_name, r["enz_ac"])
        S = load_res_embedding(model_name, r["sub_ac"])
        
        feats = make_residue_summary_features(E, S)
        feat_rows.append(feats)
    
    X_df = pd.DataFrame(feat_rows)
    
    # prefix columns
    X_df = X_df.add_prefix(f"{model_name}_res_")
    
    return X_df, meta

سلول ۸ — ساخت و ذخیره featureها برای یک مدل

In [ ]:
def save_pair_features(model_name, X_seq, X_res, meta):
    out_dir = PAIR_FEATURE_DIR / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    
    X_seq.to_parquet(out_dir / "per_sequence.features.parquet", index=False)
    X_res.to_parquet(out_dir / "per_residue.features.parquet", index=False)
    
    X_fusion = pd.concat([X_seq, X_res], axis=1)
    X_fusion.to_parquet(out_dir / "fusion.features.parquet", index=False)
    
    meta.to_csv(out_dir / "meta.csv", index=False)
    
    qc = pd.DataFrame([{
        "model": model_name,
        "n_pairs": len(meta),
        "n_seq_features": X_seq.shape[1],
        "n_res_features": X_res.shape[1],
        "n_fusion_features": X_fusion.shape[1],
        "seq_nan_count": int(X_seq.isna().sum().sum()),
        "res_nan_count": int(X_res.isna().sum().sum()),
        "fusion_nan_count": int(X_fusion.isna().sum().sum()),
        "n_positive": int((meta["label"].astype(int) == 1).sum()),
        "n_negative": int((meta["label"].astype(int) == 0).sum()),
        "n_E3": int((meta["enzyme_class"] == "E3").sum()),
        "n_DUB": int((meta["enzyme_class"] == "DUB").sum()),
    }])
    
    qc.to_csv(out_dir / "feature_qc.csv", index=False)
    
    return qc

سلول ۹ — اجرای کامل برای یک مدل

In [ ]:
def build_all_pair_features_for_model(model_name, pairs_df):
    print("="*100)
    print("Building pair features for:", model_name)
    
    X_seq, meta_seq = build_per_sequence_features_for_model(model_name, pairs_df)
    X_res, meta_res = build_per_residue_features_for_model(model_name, pairs_df)
    
    assert meta_seq["pair_id"].tolist() == meta_res["pair_id"].tolist()
    
    qc = save_pair_features(model_name, X_seq, X_res, meta_seq)
    
    display(qc)
    
    del X_seq, X_res
    gc.collect()
    
    return qc

In [ ]:
qc_t6 = build_all_pair_features_for_model(
    "esm2_t6_8M_UR50D",
    pairs_embedding_ready
)

همه مدل‌ها

In [ ]:
all_feature_qc = []

for model_name in MODELS_TO_RUN:
    qc = build_all_pair_features_for_model(model_name, pairs_embedding_ready)
    all_feature_qc.append(qc)

all_feature_qc = pd.concat(all_feature_qc, ignore_index=True)

display(all_feature_qc)

all_feature_qc.to_csv(
    QC_DIR / "pair_feature_qc_all_models.csv",
    index=False
)

QC وجود فایل‌ها و shape

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PAIR_FEATURE_DIR = DATA_ML_DIR / "pair_features"

qc_rows = []

for model_name in MODELS_TO_RUN:
    model_dir = PAIR_FEATURE_DIR / model_name
    
    files = {
        "meta": model_dir / "meta.csv",
        "seq": model_dir / "per_sequence.features.parquet",
        "res": model_dir / "per_residue.features.parquet",
        "fusion": model_dir / "fusion.features.parquet",
        "feature_qc": model_dir / "feature_qc.csv",
    }
    
    row = {"model": model_name}
    
    for k, p in files.items():
        row[f"{k}_exists"] = p.exists()
    
    if files["meta"].exists():
        meta = pd.read_csv(files["meta"], dtype=str, low_memory=False)
        row["n_meta"] = len(meta)
        row["n_pair_id_unique"] = meta["pair_id"].nunique()
        row["n_pair_id_duplicates"] = int(meta.duplicated("pair_id").sum())
        row["n_label_1"] = int((meta["label"].astype(int) == 1).sum())
        row["n_label_0"] = int((meta["label"].astype(int) == 0).sum())
        row["n_E3"] = int((meta["enzyme_class"] == "E3").sum())
        row["n_DUB"] = int((meta["enzyme_class"] == "DUB").sum())
    else:
        row["n_meta"] = np.nan
    
    for kind in ["seq", "res", "fusion"]:
        if files[kind].exists():
            X = pd.read_parquet(files[kind])
            row[f"{kind}_shape"] = str(X.shape)
            row[f"{kind}_nan"] = int(X.isna().sum().sum())
            row[f"{kind}_inf"] = int(np.isinf(X.to_numpy(dtype=np.float32)).sum())
        else:
            row[f"{kind}_shape"] = ""
            row[f"{kind}_nan"] = np.nan
            row[f"{kind}_inf"] = np.nan
    
    qc_rows.append(row)

pair_feature_file_qc = pd.DataFrame(qc_rows)
display(pair_feature_file_qc)

pair_feature_file_qc.to_csv(
    QC_DIR / "pair_feature_file_qc_all_models.csv",
    index=False
)

QC row order بین همه مدل‌ها

In [ ]:
reference_model = MODELS_TO_RUN[0]
ref_meta = pd.read_csv(
    PAIR_FEATURE_DIR / reference_model / "meta.csv",
    dtype=str,
    low_memory=False
)

order_rows = []

for model_name in MODELS_TO_RUN:
    meta = pd.read_csv(
        PAIR_FEATURE_DIR / model_name / "meta.csv",
        dtype=str,
        low_memory=False
    )
    
    same_pair_order = meta["pair_id"].tolist() == ref_meta["pair_id"].tolist()
    same_labels = meta["label"].astype(int).tolist() == ref_meta["label"].astype(int).tolist()
    same_groups = meta["group_id"].tolist() == ref_meta["group_id"].tolist()
    
    order_rows.append({
        "model": model_name,
        "same_pair_order_as_reference": same_pair_order,
        "same_labels_as_reference": same_labels,
        "same_groups_as_reference": same_groups,
        "n_rows": len(meta),
    })

pair_feature_order_qc = pd.DataFrame(order_rows)
display(pair_feature_order_qc)

pair_feature_order_qc.to_csv(
    QC_DIR / "pair_feature_order_qc_all_models.csv",
    index=False
)

 QC ستون‌های feature

In [ ]:
feature_col_qc_rows = []

for model_name in MODELS_TO_RUN:
    model_dir = PAIR_FEATURE_DIR / model_name
    
    X_seq = pd.read_parquet(model_dir / "per_sequence.features.parquet")
    X_res = pd.read_parquet(model_dir / "per_residue.features.parquet")
    X_fusion = pd.read_parquet(model_dir / "fusion.features.parquet")
    
    feature_col_qc_rows.append({
        "model": model_name,
        "seq_n_cols": X_seq.shape[1],
        "seq_unique_cols": X_seq.columns.nunique(),
        "seq_duplicate_cols": X_seq.shape[1] - X_seq.columns.nunique(),
        "res_n_cols": X_res.shape[1],
        "res_unique_cols": X_res.columns.nunique(),
        "res_duplicate_cols": X_res.shape[1] - X_res.columns.nunique(),
        "fusion_n_cols": X_fusion.shape[1],
        "fusion_unique_cols": X_fusion.columns.nunique(),
        "fusion_duplicate_cols": X_fusion.shape[1] - X_fusion.columns.nunique(),
    })

feature_col_qc = pd.DataFrame(feature_col_qc_rows)
display(feature_col_qc)

feature_col_qc.to_csv(
    QC_DIR / "pair_feature_column_qc_all_models.csv",
    index=False
)

QC مقدارهای ثابت یا مشکوک

In [ ]:
constant_feature_rows = []

for model_name in MODELS_TO_RUN:
    model_dir = PAIR_FEATURE_DIR / model_name
    
    for feature_type, fname in [
        ("per_sequence", "per_sequence.features.parquet"),
        ("per_residue", "per_residue.features.parquet"),
        ("fusion", "fusion.features.parquet"),
    ]:
        X = pd.read_parquet(model_dir / fname)
        
        stds = X.std(axis=0)
        n_constant = int((stds == 0).sum())
        
        constant_cols = stds[stds == 0].index.tolist()[:20]
        
        constant_feature_rows.append({
            "model": model_name,
            "feature_type": feature_type,
            "n_features": X.shape[1],
            "n_constant_features": n_constant,
            "constant_feature_examples": ";".join(constant_cols),
        })

constant_feature_qc = pd.DataFrame(constant_feature_rows)
display(constant_feature_qc)

constant_feature_qc.to_csv(
    QC_DIR / "pair_feature_constant_qc_all_models.csv",
    index=False
)

خلاصه نهایی روز ۸

In [ ]:
day8_final_qc = pair_feature_file_qc.merge(
    pair_feature_order_qc,
    on="model",
    how="left"
).merge(
    feature_col_qc,
    on="model",
    how="left"
)

display(day8_final_qc)

day8_final_qc.to_csv(
    QC_DIR / "day8_final_pair_feature_qc_summary.csv",
    index=False
)

print("Saved final Day 8 QC:")
print(QC_DIR / "day8_final_pair_feature_qc_summary.csv")